In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install google-generativeai chromadb --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 897.8 kB/s eta 0:00:00 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 85.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 59.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 79.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}")

✅ Gemini API key setup complete.


In [8]:
import chromadb
chroma_client = chromadb.Client()
collection = chroma_client.create_collection("faqs")

questions = [
    "How can I track my order?",
    "What is the return policy?",
    "How long does delivery take?",
]

answers = [
    "You can track your order using the tracking ID provided after purchase.",
    "Items can be returned within 7 days of delivery if unused and with tags.",
    "Delivery usually takes 3–5 business days depending on location."
]

collection.add(
    documents=answers,
    metadatas=[{"source": "FAQ"}] * len(answers),
    ids=[f"id_{i}" for i in range(len(answers))]
)


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:06<00:00, 12.1MiB/s]


In [9]:
def search_faq(query):
    result = collection.query(query_texts=[query], n_results=1)
    return result["documents"][0][0]


In [26]:
import google.generativeai as genai
def chat(query):
    faq_answer = search_faq(query)

    model = genai.GenerativeModel("gemini-2.5-flash-lite")

    response = model.generate_content(
    f"Answer the following question strictly using this FAQ text only, without rewriting or adding new information.\n\nFAQ Answer: {faq_answer}\n\nUser Question: {query}"
)


    return response.text


In [27]:
print(chat("How can I track my order?"))


You can track your order using the tracking ID provided after purchase.


In [24]:
import random

def track_order(order_id):
    statuses = ["Processing", "Packed", "Shipped", "Out for Delivery", "Delivered"]
    return f"Order {order_id} is currently: {random.choice(statuses)}"

def customer_query(msg):
    if "track" in msg.lower():
        return track_order("OD12345")
    else:
        return chat(msg)

print(customer_query("track my order"))


Order OD12345 is currently: Out for Delivery
